In [1]:
!pip install -q xgboost

In [26]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml

# Fetch the Bank Marketing dataset directly from OpenML
data = fetch_openml(name="bank-marketing", version=1, as_frame=True)

df = data.data.copy()
df["target"] = data.target.copy()

print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head(5)

Shape: (45211, 17)

First 5 rows:


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,target
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,1
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,1
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,1
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,1
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,1


In [27]:
print(f"The datatype of the columns are:\n{df.dtypes}")

The datatype of the columns are:
V1           int64
V2        category
V3        category
V4        category
V5        category
V6           int64
V7        category
V8        category
V9        category
V10          int64
V11       category
V12          int64
V13          int64
V14          int64
V15          int64
V16       category
target    category
dtype: object


In [28]:
print(f"The missing values per column is:\n{df.isna().sum()}")

The missing values per column is:
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
target    0
dtype: int64


In [20]:
print(df["target"].value_counts())

target
0    39922
1     5289
Name: count, dtype: int64


In [16]:
cat_cols = df.select_dtypes(include=["category", "object"]).columns
print(f"Categorical columns to encode: {list(cat_cols)}")

Categorical columns to encode: ['V2', 'V3', 'V4', 'V5', 'V7', 'V8', 'V9', 'V11', 'V16', 'target']


In [29]:
from sklearn.preprocessing import LabelEncoder

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))

    print(col, mapping)

V2 {'admin.': np.int64(0), 'blue-collar': np.int64(1), 'entrepreneur': np.int64(2), 'housemaid': np.int64(3), 'management': np.int64(4), 'retired': np.int64(5), 'self-employed': np.int64(6), 'services': np.int64(7), 'student': np.int64(8), 'technician': np.int64(9), 'unemployed': np.int64(10), 'unknown': np.int64(11)}
V3 {'divorced': np.int64(0), 'married': np.int64(1), 'single': np.int64(2)}
V4 {'primary': np.int64(0), 'secondary': np.int64(1), 'tertiary': np.int64(2), 'unknown': np.int64(3)}
V5 {'no': np.int64(0), 'yes': np.int64(1)}
V7 {'no': np.int64(0), 'yes': np.int64(1)}
V8 {'no': np.int64(0), 'yes': np.int64(1)}
V9 {'cellular': np.int64(0), 'telephone': np.int64(1), 'unknown': np.int64(2)}
V11 {'apr': np.int64(0), 'aug': np.int64(1), 'dec': np.int64(2), 'feb': np.int64(3), 'jan': np.int64(4), 'jul': np.int64(5), 'jun': np.int64(6), 'mar': np.int64(7), 'may': np.int64(8), 'nov': np.int64(9), 'oct': np.int64(10), 'sep': np.int64(11)}
V16 {'failure': np.int64(0), 'other': np.int64

In [30]:
df.head(5)

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,target
0,58,4,1,2,0,2143,1,0,2,5,8,261,1,-1,0,3,0
1,44,9,2,1,0,29,1,0,2,5,8,151,1,-1,0,3,0
2,33,2,1,1,0,2,1,1,2,5,8,76,1,-1,0,3,0
3,47,1,1,3,0,1506,1,0,2,5,8,92,1,-1,0,3,0
4,33,11,2,3,0,1,0,0,2,5,8,198,1,-1,0,3,0


In [49]:
import pandas as pd

# Configuration
SAMPLES_PER_BATCH = 3000
RANDOM_SEED = 42

# Label encoder mapping
month_mapping = {
    4: "jan",
    3: "feb",
    7: "mar",
    0: "apr",
    8: "may",
    6: "jun",
    5: "jul",
    1: "aug",
    11: "sep",
    10: "oct",
    9: "nov",
    2: "dec"
}

# Create batches in chronological month order
batches = []

for batch_num, (month_code, month_name) in enumerate(month_mapping.items(), start=1):

    # Select only rows belonging to this month
    month_data = df[df["V11"] == month_code].copy()

    # Take up to 3,000 rows
    n_samples = min(SAMPLES_PER_BATCH, len(month_data))

    batch = month_data.sample(
        n=n_samples,
        random_state=RANDOM_SEED
    ).reset_index(drop=True)

    batches.append(batch)

    print(
        f"Batch {batch_num}: {month_name.upper()} : {len(batch)} rows"
    )

print(f"\nCreated {len(batches)} monthly batches")


Batch 1: JAN : 1403 rows
Batch 2: FEB : 2649 rows
Batch 3: MAR : 477 rows
Batch 4: APR : 2932 rows
Batch 5: MAY : 3000 rows
Batch 6: JUN : 3000 rows
Batch 7: JUL : 3000 rows
Batch 8: AUG : 3000 rows
Batch 9: SEP : 579 rows
Batch 10: OCT : 738 rows
Batch 11: NOV : 3000 rows
Batch 12: DEC : 214 rows

Created 12 monthly batches


In [52]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from xgboost import XGBClassifier

# 1. Separate features and target
feature_names = [col for col in df.columns if col != "target"]

# We use Batch 1 (JANUARY) as our training data (Day 0)
baseline_data = batches[0].copy()

X_jan = baseline_data[feature_names]
y_jan = baseline_data["target"]

# 2. Split January into Training and Validation
X_train, X_val, y_train, y_val = train_test_split(
    X_jan, y_jan,
    test_size=0.2,
    random_state=42,
    stratify=y_jan
)

# 3. Train XGBoost Classifier
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

# 4. Evaluate baseline performance on held-out January validation data
val_preds = model.predict(X_val)
val_probs = model.predict_proba(X_val)[:, 1]

baseline_metrics = {
    "accuracy": accuracy_score(y_val, val_preds),
    "f1": f1_score(y_val, val_preds, zero_division=0),
    "roc_auc": roc_auc_score(y_val, val_probs)
}

print("BASELINE MODEL (VERSION 0) TRAINED ON JANUARY")
print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Baseline Accuracy:  {baseline_metrics['accuracy']:.4f}")
print(f"Baseline F1-Score:  {baseline_metrics['f1']:.4f}")
print(f"Baseline ROC-AUC:   {baseline_metrics['roc_auc']:.4f}")

BASELINE MODEL (VERSION 0) TRAINED ON JANUARY
Training samples:   1122
Validation samples: 281
Baseline Accuracy:  0.9359
Baseline F1-Score:  0.5909
Baseline ROC-AUC:   0.9445


In [53]:
# 1. Reference features (all January features)
reference_features = X_jan.copy()

# 2. Reference prediction probabilities (how confident the model was on January data)
reference_probabilities = model.predict_proba(X_jan)[:, 1]

print("Reference profile locked!")
print(f"Reference data shape: {reference_features.shape}")
print(f"Average prediction probability in Jan: {reference_probabilities.mean():.4f}")
print(f"Median prediction probability in Jan:  {np.median(reference_probabilities):.4f}")

Reference profile locked!
Reference data shape: (1403, 16)
Average prediction probability in Jan: 0.0984
Median prediction probability in Jan:  0.0087


In [54]:
# Evaluate the January model directly on February data
feb_data = batches[1].copy()
X_feb = feb_data[feature_names]
y_feb = feb_data["target"]

feb_preds = model.predict(X_feb)
feb_probs = model.predict_proba(X_feb)[:, 1]

feb_acc = accuracy_score(y_feb, feb_preds)
feb_f1 = f1_score(y_feb, feb_preds, zero_division=0)
feb_auc = roc_auc_score(y_feb, feb_probs)

print("  JANUARY MODEL EVALUATED ON FEBRUARY DATA")
print(f"{'Metric':<15} {'January (Baseline)':<20} {'February':<15} {'Change':<10}")
print("-" * 55)
print(f"{'Accuracy':<15} {baseline_metrics['accuracy']:<20.4f} {feb_acc:<15.4f} {feb_acc - baseline_metrics['accuracy']:+.4f}")
print(f"{'F1-Score':<15} {baseline_metrics['f1']:<20.4f} {feb_f1:<15.4f} {feb_f1 - baseline_metrics['f1']:+.4f}")
print(f"{'ROC-AUC':<15} {baseline_metrics['roc_auc']:<20.4f} {feb_auc:<15.4f} {feb_auc - baseline_metrics['roc_auc']:+.4f}")
print("-" * 55)

  JANUARY MODEL EVALUATED ON FEBRUARY DATA
Metric          January (Baseline)   February        Change    
-------------------------------------------------------
Accuracy        0.9359               0.6655          -0.2704
F1-Score        0.5909               0.3864          -0.2045
ROC-AUC         0.9445               0.7104          -0.2342
-------------------------------------------------------
